<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/Manual_regex_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Manual práctico: expresiones regulares con pandas
## De los primeros patrones a la limpieza de datos
Material para estudiantes de Big Data. Los datos son ficticios y los ejemplos se pueden ejecutar sin descargar archivos ni usar credenciales.

**Cómo usarlo en Colab:** Archivo → Subir cuaderno → seleccionar este `.ipynb`. Ejecuta las celdas en orden. Prueba a cambiar los patrones y compara los resultados.

**Objetivos:** interpretar patrones; elegir el método de pandas; filtrar, validar, extraer y transformar texto; manejar nulos; construir un pequeño proceso de limpieza.

**Recorrido:** 1–4 fundamentos; 5–10 métodos; 11–15 técnicas avanzadas; 16 caso práctico; 17–19 rendimiento, errores y ejercicios; 20 referencias.

## 1. ¿Qué es una expresión regular?
Es un patrón que describe qué secuencias de caracteres buscamos. Por ejemplo, `PED-` es texto literal y `[0-9]{4}` representa cuatro dígitos ASCII. Juntos reconocen códigos como `PED-0042`.

Pandas aplica estos patrones a una columna mediante `.str`: `df["columna"].str.metodo(patron)`. El resultado puede ser una máscara booleana, una columna transformada o varias columnas nuevas.

In [ ]:
import pandas as pd
import re
from IPython.display import display
print("Versión de pandas:", pd.__version__)
# Elegimos almacenamiento Python para los patrones avanzados de este manual.
texto = pd.StringDtype(storage="python")
s = pd.Series(["PED-0042", "Error PED-0100", "sin pedido", None], dtype=texto)
display(pd.DataFrame({"texto": s, "contiene_pedido": s.str.contains(r"PED-[0-9]{4}", na=False)}))

Versión de pandas: 2.2.3


,texto,contiene_pedido
0,PED-0042,True
1,Error PED-0100,True
2,sin pedido,False
3,<NA>,False


## 2. Cadenas raw: por qué escribimos `r"..."`
Python interpreta escapes en una cadena normal. Por ejemplo, `"\n"` es un salto de línea y `"\b"` es un carácter de retroceso. La forma raw conserva las barras para que las interprete el motor de regex: `r"\bventa\b"` busca la palabra completa.

La `r` pertenece a la sintaxis de Python; no forma parte del patrón. También puedes escribir `"\\d+"`, pero `r"\d+"` es más legible.

In [ ]:
print(repr("\b"), repr(r"\b"))
display(pd.Series(["venta", "preventas", "una venta"], dtype=texto).str.contains(r"\bventa\b", na=False))

'\x08' '\\b'


0     True
1    False
2     True
dtype: boolean

## 3. Chuleta de sintaxis

| Patrón | Significado | Ejemplo |
|---|---|---|
| `abc` | Texto literal | `abc` |
| `.` | Un carácter, salvo salto de línea por defecto | `a.c` |
| `\.` | Punto literal | `archivo\.csv` |
| `[abc]` | Un carácter del conjunto | `a`, `b` o `c` |
| `[A-Z]` | Una mayúscula ASCII | `M` |
| `[^0-9]` | Un carácter que no sea un dígito ASCII | `x` |
| `\d` / `\D` | Dígito Unicode / no dígito | `7` |
| `\w` / `\W` | Letra, dígito o `_` Unicode / lo contrario | `ñ` |
| `\s` / `\S` | Espacio en blanco / lo contrario | espacio, tabulación |
| `*` | Cero o más repeticiones | `ab*`: `a`, `ab`, `abb` |
| `+` | Una o más repeticiones | `ab+`: `ab`, `abb` |
| `?` | Cero o una repetición | `colou?r` |
| `{3}` | Exactamente tres | `[0-9]{3}` |
| `{2,4}` | Entre dos y cuatro | `[A-Z]{2,4}` |
| `{2,}` | Dos o más | `[0-9]{2,}` |
| `^` / `$` | Inicio / final, con matices en texto multilínea | `^PED-` |
| `\b` | Límite de palabra | `\bIA\b` |
| `A\|B`* | Alternativa | `rojo` o `azul` |
| `(abc)` | Grupo que captura | Extraer `abc` |
| `(?:abc)` | Agrupar sin capturar | Repetir una alternativa |
| `(?P<nombre>...)` | Grupo con nombre | Columna `nombre` |

*La barra vertical de alternativa se escribe sin barra invertida en el código: `r"rojo|azul"`. En esta tabla se escapa para no romper las columnas.

**Dos matices:** `\d` incluye dígitos de otros alfabetos; usa `[0-9]` para formatos ASCII. `$` puede coincidir antes de un salto de línea final; usa `fullmatch` para validar toda la cadena.

In [ ]:
muestras = pd.Series(["123", "１２３", "12a", "", None], dtype=texto)
display(pd.DataFrame({"dato": muestras,
                      "digitos_unicode": muestras.str.fullmatch(r"\d+", na=False),
                      "digitos_ASCII": muestras.str.fullmatch(r"[0-9]+", na=False)}))

,dato,digitos_unicode,digitos_ASCII
0,123,True,True
1,１２３,True,False
2,12a,False,False
3,,False,False
4,<NA>,False,False


## 4. Datos de trabajo y valores ausentes
Usaremos `StringDtype(storage="python")`: conserva los nulos y permite usar el motor Python en los ejemplos avanzados. `astype(str)` puede convertir ausencias en textos como `"None"` o `"nan"`; no es una limpieza de nulos.

En una máscara de filtrado, `na=False` indica que una ausencia no cumple la condición. Esa decisión no convierte el dato ausente en un valor válido: para revisar calidad, comprueba `isna()` por separado.

In [ ]:
df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "descripcion": ["Portátil Lenovo 16 GB", "ratón USB", "Monitor 27 pulgadas", None, "SSD 512 GB + RAM 8 GB"],
    "email": [" ANA@EJEMPLO.ES ", "luis@example.com", "mal@@correo", None, "eva@ejemplo.es"],
    "pedido": ["PED-2026-0001", "PED-2026-0002", "ped-2026-0003", None, "PED-26-0005"],
    "precio": ["1.249,90 €", "19,95 €", "229,00 €", None, "89,50 €"],
    "telefono": ["+34 612 345 678", "699-111-222", "12345", None, "600 000 001"]
})
for columna in df.columns.drop("id"):
    df[columna] = df[columna].astype(texto)
display(df)

,id,descripcion,email,pedido,precio,telefono
0,1,Portátil Lenovo 16 GB,ANA@EJEMPLO.ES,PED-2026-0001,"1.249,90 €",+34 612 345 678
1,2,ratón USB,luis@example.com,PED-2026-0002,"19,95 €",699-111-222
2,3,Monitor 27 pulgadas,mal@@correo,ped-2026-0003,"229,00 €",12345
3,4,<NA>,<NA>,<NA>,<NA>,<NA>
4,5,SSD 512 GB + RAM 8 GB,eva@ejemplo.es,PED-26-0005,"89,50 €",600 000 001


## 5. Buscar y filtrar: `str.contains`
Busca una coincidencia en cualquier posición. Devuelve una máscara para seleccionar filas. `case=False` ignora mayúsculas, pero no elimina diferencias de acentos.

`regex=False` busca texto literal. Es apropiado si buscas símbolos como `+`, `.` o paréntesis y no quieres que actúen como instrucciones.

In [ ]:
mascara = df["descripcion"].str.contains(r"portátil|monitor", case=False, na=False)
display(df.loc[mascara, ["id", "descripcion"]])  # ids 1 y 3
display(df.loc[df["descripcion"].str.contains("+", regex=False, na=False), ["id", "descripcion"]])
# Palabra completa: RAM no debe reconocer PROGRAMAR.
display(pd.Series(["RAM 8 GB", "PROGRAMAR", "ram"], dtype=texto)
        .str.contains(r"\bRAM\b", case=False, na=False))

,id,descripcion
0,1,Portátil Lenovo 16 GB
2,3,Monitor 27 pulgadas


,id,descripcion
4,5,SSD 512 GB + RAM 8 GB


0     True
1    False
2     True
dtype: boolean

### Combinar condiciones y buscar texto introducido por una persona
Las máscaras usan `&`, `|` y `~`, con paréntesis. No uses `and` u `or` entre Series.
Si necesitas insertar texto literal en una regex, `re.escape` escapa sus metacaracteres.

In [ ]:
es_almacenamiento = df["descripcion"].str.contains(r"\bSSD\b", na=False)
tiene_gb = df["descripcion"].str.contains(r"[0-9]+\s*GB", na=False)
display(df.loc[es_almacenamiento & tiene_gb, ["id", "descripcion"]])
termino = "C++"
lenguajes = pd.Series(["Curso C++", "Curso Python", "Curso C"], dtype=texto)
display(lenguajes.str.contains(re.escape(termino), na=False))
# Para una búsqueda literal simple: lenguajes.str.contains(termino, regex=False, na=False)

,id,descripcion
4,5,SSD 512 GB + RAM 8 GB


0     True
1    False
2    False
dtype: boolean

## 6. `contains`, `match` y `fullmatch`

| Método | Pregunta |
|---|---|
| `contains` | ¿Aparece el patrón en algún lugar? |
| `match` | ¿El texto empieza coincidiendo con el patrón? |
| `fullmatch` | ¿Todo el texto cumple el patrón? |

Para un código válido necesitas `fullmatch`; `match` podría aceptar un sufijo no previsto.

In [ ]:
codigos = pd.Series(["PED-123", "texto PED-123", "PED-123-extra", None], dtype=texto)
patron = r"PED-[0-9]{3}"
display(pd.DataFrame({"texto": codigos,
    "contains": codigos.str.contains(patron, na=False),
    "match": codigos.str.match(patron, na=False),
    "fullmatch": codigos.str.fullmatch(patron, na=False)}))

,texto,contains,match,fullmatch
0,PED-123,True,True,True
1,texto PED-123,True,False,False
2,PED-123-extra,True,True,False
3,<NA>,False,False,False


## 7. Validar formatos
`PED-[0-9]{4}-[0-9]{4}` describe prefijo, año y número de pedido. No comprueba que el año esté dentro de un intervalo de negocio.

La regex de correo del ejemplo es una comprobación básica: no implementa todos los estándares de correo ni verifica que la dirección exista. Del mismo modo, una fecha bien escrita puede ser imposible; después de extraerla hay que convertirla con `pd.to_datetime`.

In [ ]:
df["pedido_valido"] = df["pedido"].str.fullmatch(r"PED-[0-9]{4}-[0-9]{4}", na=False)
correo = df["email"].str.strip().str.lower()
df["email_valido"] = correo.str.fullmatch(r"[^\s@]+@[^\s@]+\.[^\s@]+", na=False)
display(df[["id", "pedido", "pedido_valido", "email", "email_valido"]])

,id,pedido,pedido_valido,email,email_valido
0,1,PED-2026-0001,True,ANA@EJEMPLO.ES,True
1,2,PED-2026-0002,True,luis@example.com,True
2,3,ped-2026-0003,False,mal@@correo,False
3,4,<NA>,False,<NA>,False
4,5,PED-26-0005,False,eva@ejemplo.es,True


## 8. Extraer: `str.extract`
Los paréntesis indican qué conservar. `extract` toma la primera coincidencia de cada fila; sin coincidencia devuelve valores ausentes. Con varios grupos obtienes varias columnas.

Un grupo con nombre `(?P<anio>[0-9]{4})` crea una columna llamada `anio`. Para obtener una Series con un solo grupo, usa `expand=False`. Las cifras extraídas siguen siendo texto: conviértelas después.

In [ ]:
partes = df["pedido"].str.extract(r"^PED-(?P<anio>[0-9]{4})-(?P<numero>[0-9]{4})$")
partes["anio"] = pd.to_numeric(partes["anio"], errors="coerce").astype("Int64")
# Conservamos numero como texto: 0001 es un identificador, no una cantidad.
display(partes)
capacidad = df["descripcion"].str.extract(r"([0-9]+)\s*GB", expand=False)
display(pd.to_numeric(capacidad, errors="coerce").astype("Int64"))

,anio,numero
0,2026,0001
1,2026,0002
2,<NA>,<NA>
3,<NA>,<NA>
4,<NA>,<NA>


0      16
1    <NA>
2    <NA>
3    <NA>
4     512
Name: descripcion, dtype: Int64

## 9. Todas las coincidencias: `findall`, `extractall` y `count`
`findall` produce listas por fila; `extractall` produce una fila por coincidencia con un MultiIndex: índice original y número de coincidencia. Las filas sin coincidencias no aparecen en `extractall`.

`count` cuenta coincidencias no solapadas. Si `findall` usa un grupo capturante devuelve lo capturado; con varios grupos devuelve tuplas. Sin grupos devuelve las coincidencias completas.

In [ ]:
display(df["descripcion"].str.findall(r"[0-9]+\s*GB"))
display(df["descripcion"].str.count(r"[0-9]+\s*GB").fillna(0).astype("Int64"))
capacidades = df["descripcion"].str.extractall(r"(?P<cantidad>[0-9]+)\s*(?P<unidad>GB)")
display(capacidades)
display(capacidades.reset_index().rename(columns={"level_0": "indice_origen"}))
# explode conserva el índice de origen; una lista vacía genera un nulo.
display(df["descripcion"].str.findall(r"[0-9]+\s*GB").explode().dropna())

0           [16 GB]
1                []
2                []
3              <NA>
4    [512 GB, 8 GB]
Name: descripcion, dtype: object

0    1
1    0
2    0
3    0
4    2
Name: descripcion, dtype: Int64

cantidad unidad
  match                
0 0           16     GB
4 0          512     GB
  1            8     GB

,indice_origen,match,cantidad,unidad
0,0,0,16,GB
1,4,0,512,GB
2,4,1,8,GB


0     16 GB
4    512 GB
4      8 GB
Name: descripcion, dtype: object

## 10. Limpiar y reemplazar: `str.replace`
Usa `regex=True` para ejecutar un patrón y `regex=False` para una sustitución literal. El resultado es una nueva Series; asígnala para conservarla.

Proceso habitual: recortar extremos con `strip`, unificar espacios con `\s+` y después normalizar el formato. No elimines todos los caracteres no numéricos de un precio: perderías los decimales.

In [ ]:
nombres = pd.Series(["  Ana   García ", "Luis\tPérez", None], dtype=texto)
display(nombres.str.strip().str.replace(r"\s+", " ", regex=True))
# Convención española en este conjunto: punto de miles y coma decimal.
precios_limpios = (df["precio"].str.replace(r"[€\s]", "", regex=True)
                  .str.replace(".", "", regex=False)
                  .str.replace(",", ".", regex=False))
df["precio_eur"] = pd.to_numeric(precios_limpios, errors="coerce")
display(df[["precio", "precio_eur"]])
assert abs(df.loc[0, "precio_eur"] - 1249.90) < 1e-9

0    Ana García
1    Luis Pérez
2          <NA>
dtype: string

,precio,precio_eur
0,"1.249,90 €",1249.9
1,"19,95 €",19.95
2,"229,00 €",229.0
3,<NA>,<NA>
4,"89,50 €",89.5


### Reutilizar grupos en una sustitución
En el reemplazo, `\g<nombre>` recupera el texto capturado. Esto transforma un formato; no valida el calendario.

In [ ]:
fechas = pd.Series(["12/10/2026", "31/02/2026", "sin fecha", None], dtype=texto)
iso = fechas.str.replace(r"(?P<dia>[0-9]{2})/(?P<mes>[0-9]{2})/(?P<anio>[0-9]{4})",
                        r"\g<anio>-\g<mes>-\g<dia>", regex=True)
display(pd.DataFrame({"original": fechas, "iso": iso,
                      "fecha_real": pd.to_datetime(iso, format="%Y-%m-%d", errors="coerce")}))

,original,iso,fecha_real
0,12/10/2026,2026-10-12,2026-10-12
1,31/02/2026,2026-02-31,NaT
2,sin fecha,sin fecha,NaT
3,<NA>,<NA>,NaT


## 11. Separar: `str.split`
`[,;|]` permite separar por coma, punto y coma o barra vertical. Dentro de una clase de caracteres, `|` es literal. `expand=True` crea columnas; sin esa opción obtienes listas. `n=1` limita a una separación.

Especifica `regex=True` o `False` para expresar tu intención. Usa grupos no capturantes para separadores compuestos; los grupos capturantes pueden introducir el separador en el resultado.

In [ ]:
etiquetas = pd.Series(["python, pandas; numpy|spark", "sql; cloud", None], dtype=texto)
display(etiquetas.str.split(r"\s*[,;|]\s*", regex=True))
display(etiquetas.str.split(r"\s*[,;|]\s*", regex=True, expand=True))
display(pd.Series(["clave=valor=extra"], dtype=texto).str.split("=", n=1, regex=False, expand=True))

0    [python, pandas, numpy, spark]
1                      [sql, cloud]
2                              <NA>
dtype: object

,0,1,2,3
0,python,pandas,numpy,spark
1,sql,cloud,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>


,0,1
0,clave,valor=extra


## 12. Grupos, alternativas y cuantificadores
`(?:SSD|RAM)` agrupa alternativas sin crear una columna capturada. Los cuantificadores suelen ser voraces: toman todo lo posible. Añadir `?` hace la repetición mínima, por ejemplo `.*?`.

En un texto entre corchetes, `\[[^\]]*\]` también limita explícitamente los caracteres permitidos. Para formatos complejos, como HTML, JSON o CSV, usa un parser apropiado.

In [ ]:
equipos = pd.Series(["SSD 512 GB", "RAM 16 GB", "Disco 1 TB"], dtype=texto)
display(equipos.str.extract(r"(?P<tipo>SSD|RAM)\s+(?P<gb>[0-9]+)\s+GB"))
mensajes = pd.Series(["[INFO] proceso [OK]"], dtype=texto)
display(pd.DataFrame({"voraz": mensajes.str.findall(r"\[.*\]"),
                      "minimo": mensajes.str.findall(r"\[.*?\]")}))

,tipo,gb
0,SSD,512
1,RAM,16
2,<NA>,<NA>


,voraz,minimo
0,[[INFO] proceso [OK]],"[[INFO], [OK]]"


## 13. Lookahead y lookbehind
Comprueban contexto sin incorporarlo a la coincidencia.

| Construcción | Condición |
|---|---|
| `(?=...)` | Debe aparecer después |
| `(?!...)` | No debe aparecer después |
| `(?<=...)` | Debe aparecer antes |
| `(?<!...)` | No debe aparecer antes |

En `re`, el lookbehind exige anchura fija: `(?<=EUR )` sirve, pero `(?<=EUR\s+)` no. En ese caso suele ser más claro capturar `EUR\s+([0-9]+)`.

In [ ]:
importes = pd.Series(["Total 25 EUR", "Total 15 USD", "EUR 40"], dtype=texto)
display(importes.str.findall(r"[0-9]+(?=\sEUR\b)"))
display(importes.str.findall(r"(?<=EUR )[0-9]+"))
productos = pd.Series(["PRO-123", "PRO-000", "XPRO-456"], dtype=texto)
display(productos.str.fullmatch(r"PRO-(?!000)[0-9]{3}", na=False))

0    [25]
1      []
2      []
dtype: object

0      []
1      []
2    [40]
dtype: object

0     True
1    False
2    False
dtype: boolean

## 14. Flags y patrones legibles
`re.IGNORECASE` ignora mayúsculas; `re.MULTILINE` permite que `^` y `$` actúen por línea; `re.DOTALL` permite que `.` incluya saltos; `re.VERBOSE` permite comentarios y espacios de formato.

Un patrón compilado puede reutilizarse. Coloca las flags en `re.compile`; no añadas a la vez `case=False` o flags incompatibles en el método. Para los ejemplos avanzados mantenemos almacenamiento Python: otros motores, como los usados en ciertas operaciones con Arrow, tienen compatibilidad diferente.

In [ ]:
patron_pedido = re.compile(r"""
    PED-                  # prefijo
    (?P<anio>[0-9]{4})-    # año
    (?P<numero>[0-9]{4})   # secuencia
""", flags=re.VERBOSE | re.IGNORECASE)
display(df["pedido"].str.fullmatch(patron_pedido, na=False))
logs_multilinea = pd.Series(["INFO inicio\nERROR fallo\nINFO fin"], dtype=texto)
display(logs_multilinea.str.findall(r"^ERROR.*$", flags=re.MULTILINE))

0     True
1     True
2     True
3    False
4    False
Name: pedido, dtype: boolean

0    [ERROR fallo]
dtype: object

## 15. Acentos y Unicode
Ignorar mayúsculas no equivale a ignorar tildes. Puedes descomponer los caracteres con NFD y retirar las marcas combinantes del bloque U+0300–U+036F. En los ejemplos españoles, esto también convierte `ñ` en `n`: conserva la columna original si esa diferencia importa.

In [ ]:
ciudades = pd.Series(["Málaga", "MALAGA", "A Coruña", None], dtype=texto)
normalizadas = (ciudades.str.normalize("NFD")
                .str.replace(r"[\u0300-\u036f]", "", regex=True).str.casefold())
display(pd.DataFrame({"original": ciudades, "busqueda": normalizadas,
                      "es_malaga": normalizadas.str.contains("malaga", regex=False, na=False)}))

,original,busqueda,es_malaga
0,Málaga,malaga,True
1,MALAGA,malaga,True
2,A Coruña,a coruna,False
3,<NA>,<NA>,False


## 16. Caso práctico: limpiar un fichero de clientes y pedidos
Creamos un resultado nuevo y conservamos los datos originales. Distinguimos ausencias de formatos incorrectos. Para este ejercicio, los teléfonos son móviles españoles: nueve dígitos que empiezan por 6 o 7, con prefijo opcional `+34`. Esta regla simplificada no consulta asignaciones reales.

Después de quitar separadores, retiramos el prefijo solo si el resto tiene nueve dígitos. Un error de formato no se debe arreglar inventando datos.

In [ ]:
limpio = df.copy()
limpio["email_limpio"] = limpio["email"].str.strip().str.lower()
limpio["pedido_limpio"] = limpio["pedido"].str.strip().str.upper()
limpio["pedido_ok"] = limpio["pedido_limpio"].str.fullmatch(r"PED-[0-9]{4}-[0-9]{4}", na=False)
limpio["email_ok"] = limpio["email_limpio"].str.fullmatch(r"[^\s@]+@[^\s@]+\.[^\s@]+", na=False)
limpio["telefono_limpio"] = (limpio["telefono"].str.replace(r"[\s()-]", "", regex=True)
                              .str.replace(r"^\+34(?=[0-9]{9}$)", "", regex=True))
limpio["telefono_ok"] = limpio["telefono_limpio"].str.fullmatch(r"[67][0-9]{8}", na=False)
limpio["registro_ok"] = limpio["pedido_ok"] & limpio["email_ok"] & limpio["telefono_ok"] & limpio["precio_eur"].notna()
display(limpio[["id", "pedido_limpio", "email_limpio", "telefono_limpio", "precio_eur", "registro_ok"]])
calidad = pd.DataFrame({
    "ausentes": limpio[["pedido", "email", "telefono"]].isna().sum(),
    "formato_incorrecto": pd.Series({
        "pedido": ((~limpio["pedido_ok"]) & limpio["pedido"].notna()).sum(),
        "email": ((~limpio["email_ok"]) & limpio["email"].notna()).sum(),
        "telefono": ((~limpio["telefono_ok"]) & limpio["telefono"].notna()).sum()
    })
})
display(calidad)
assert limpio.loc[limpio["registro_ok"], "id"].tolist() == [1, 2]
# Para exportar, descomenta:
# limpio.to_csv("pedidos_limpios.csv", index=False)

,id,pedido_limpio,email_limpio,telefono_limpio,precio_eur,registro_ok
0,1,PED-2026-0001,ana@ejemplo.es,612345678,1249.9,True
1,2,PED-2026-0002,luis@example.com,699111222,19.95,True
2,3,PED-2026-0003,mal@@correo,12345,229.0,False
3,4,<NA>,<NA>,<NA>,<NA>,False
4,5,PED-26-0005,eva@ejemplo.es,600000001,89.5,False


,ausentes,formato_incorrecto
pedido,1,1
email,1,1
telefono,1,1


## 17. Aplicación a Big Data y rendimiento
Una regex trabaja sobre el contenido de cada cadena. El coste depende del número de filas, la longitud del texto, el patrón y el motor. Una operación `.str` no implica automáticamente ejecución paralela ni que siempre supere a cualquier alternativa.

- Para texto literal, usa `regex=False`; para prefijos o sufijos, considera `startswith` y `endswith`.
- Selecciona las columnas necesarias y evita repetir la misma limpieza.
- Evita patrones con repeticiones ambiguas anidadas como `(a+)+$`: algunos motores pueden explorar muchísimas combinaciones en casos desfavorables.
- Mide con tus propios datos. Compilar facilita reutilizar el patrón, pero no garantiza una mejora: Python ya mantiene una caché.
- Si el fichero no cabe en memoria, procesa bloques con `pd.read_csv(..., chunksize=...)`. No acumules todos los bloques para concatenarlos después.
- En Spark o motores SQL, comprueba su sintaxis: no todos usan el motor `re` de Python.

Plantilla de lectura por bloques, para adaptar a tu CSV:

```python
for bloque in pd.read_csv("clientes.csv", chunksize=100_000,
                          dtype={"email": "string"}):
    bloque["email"] = bloque["email"].str.strip().str.lower()
    validos = bloque["email"].str.fullmatch(patron_email, na=False)
    # Agregar métricas o escribir el resultado de este bloque.
```

In [ ]:
from timeit import repeat
serie_grande = pd.Series(["pedido PED-2026-0001", "sin pedido"] * 25000, dtype=texto)
def busqueda_literal():
    return serie_grande.str.contains("PED-", regex=False, na=False)
def busqueda_regex():
    return serie_grande.str.contains(r"PED-", regex=True, na=False)
assert busqueda_literal().equals(busqueda_regex())
for nombre, funcion in [("literal", busqueda_literal), ("regex", busqueda_regex)]:
    tiempos = repeat(funcion, number=1, repeat=3)
    print(nombre, "mejor tiempo:", round(min(tiempos), 4), "segundos")
# Esta medida solo compara estos datos, este patrón y este entorno.

literal mejor tiempo: 0.0049 segundos
regex mejor tiempo: 0.0075 segundos


## 18. Errores frecuentes y método de depuración

| Problema | Solución |
|---|---|
| Un punto reconoce cualquier carácter | Usa `\.` o `regex=False` |
| Un prefijo válido acepta basura al final | Usa `fullmatch` |
| Falla un filtro con datos ausentes | Explicita `na=False`; revisa `isna` |
| `extract` exige grupos de captura | Añade `(...)` o `(?P<nombre>...)` |
| `contains` avisa de grupos de captura | Si no vas a extraer, usa `(?:...)` |
| El reemplazo no interpreta el patrón | Indica `regex=True` |
| Se han perdido ceros iniciales | Conserva identificadores como texto |
| `case=False` no reconoce una variante sin tilde | Normaliza acentos según tu regla de negocio |
| Un patrón avanzado falla según el tipo de columna | Revisa motor y almacenamiento; prueba Python |
| Una regex encuentra una fecha imposible | Convierte con `to_datetime(errors="coerce")` |

**Depura con seis entradas:** una válida, una inválida, una vacía, una ausente, una con texto extra y una con caracteres inesperados. Primero prueba una Series pequeña; después aplica el patrón al conjunto completo.

In [ ]:
pruebas = pd.Series(["PED-2026-0001", "PED-26-1", "", None,
                     "PED-2026-0001extra", "PED-２０２６-0001"], dtype=texto)
obtenido = pruebas.str.fullmatch(r"PED-[0-9]{4}-[0-9]{4}", na=False)
assert obtenido.tolist() == [True, False, False, False, False, False]
display(pd.DataFrame({"entrada": pruebas, "valida": obtenido}))

,entrada,valida
0,PED-2026-0001,True
1,PED-26-1,False
2,,False
3,<NA>,False
4,PED-2026-0001extra,False
5,PED-２０２６-0001,False


## 19. Ejercicios progresivos
Intenta resolverlos antes de ejecutar las soluciones.

1. Selecciona las descripciones con `GB` como palabra completa. Esperado: ids 1 y 5.
2. Valida códigos postales de cinco dígitos ASCII. Esto valida formato, no existencia.
3. Extrae marca y capacidad de `Lenovo:16GB` y `Dell:32GB` en dos columnas.
4. Convierte `"  Data    Science  "` en `"Data Science"`.
5. Extrae todos los hashtags de cada mensaje, incluyendo letras acentuadas.
6. Anonimiza conservando el dominio de los correos que cumplen el formato básico.
7. Extrae fecha, nivel y mensaje de un log; convierte la fecha y cuenta niveles.
8. Clasifica cada pedido como `ausente`, `válido` o `formato incorrecto`.

In [ ]:
# Espacio para tus respuestas.
# Crea tus variables aquí y contrasta con las soluciones siguientes.

### Soluciones 1–4

In [ ]:
display(df.loc[df["descripcion"].str.contains(r"\bGB\b", na=False), ["id", "descripcion"]])
cp = pd.Series(["29001", "08001", "1234", "ABCDE", None], dtype=texto)
display(cp.str.fullmatch(r"[0-9]{5}", na=False))
equipos_ej = pd.Series(["Lenovo:16GB", "Dell:32GB", "dato incorrecto"], dtype=texto)
display(equipos_ej.str.extract(r"^(?P<marca>[A-Za-z]+):(?P<gb>[0-9]+)GB$"))
display(pd.Series(["  Data    Science  "], dtype=texto).str.strip().str.replace(r"\s+", " ", regex=True))

,id,descripcion
0,1,Portátil Lenovo 16 GB
4,5,SSD 512 GB + RAM 8 GB


0     True
1     True
2    False
3    False
4    False
dtype: boolean

,marca,gb
0,Lenovo,16
1,Dell,32
2,<NA>,<NA>


0    Data Science
dtype: string

### Soluciones 5–6
En este ejercicio, un hashtag empieza por letra o `_` y continúa con caracteres de palabra. El patrón excluye un primer dígito. La anonimización didáctica del correo no anonimiza otras columnas del conjunto.

In [ ]:
mensajes_ej = pd.Series(["Aprendemos #Python y #Análisis", "#BigData #2026", None], dtype=texto)
display(mensajes_ej.str.findall(r"#([^\W\d]\w*)"))
patron_email = r"[^\s@]+@[^\s@]+\.[^\s@]+"
validos_email = correo.str.fullmatch(patron_email, na=False)
anonimo = correo.where(validos_email).str.replace(r"^[^@]+", "***", regex=True)
display(pd.DataFrame({"correo": correo, "anonimo": anonimo}))

0    [Python, Análisis]
1             [BigData]
2                  <NA>
dtype: object

,correo,anonimo
0,ana@ejemplo.es,***@ejemplo.es
1,luis@example.com,***@example.com
2,mal@@correo,<NA>
3,<NA>,<NA>
4,eva@ejemplo.es,***@ejemplo.es


### Soluciones 7–8

In [ ]:
logs = pd.Series(["2026-10-12 INFO Inicio", "2026-10-12 ERROR Fallo BD",
                  "2026-02-31 WARN Revisar fecha", "texto libre", None], dtype=texto)
tabla_log = logs.str.extract(r"^(?P<fecha>[0-9]{4}-[0-9]{2}-[0-9]{2})\s+(?P<nivel>INFO|WARN|ERROR)\s+(?P<mensaje>.+)$")
tabla_log["fecha"] = pd.to_datetime(tabla_log["fecha"], format="%Y-%m-%d", errors="coerce")
display(tabla_log)
display(tabla_log["nivel"].value_counts())
estado = pd.Series("formato incorrecto", index=df.index, dtype=texto)
estado.loc[df["pedido_valido"]] = "válido"
estado.loc[df["pedido"].isna()] = "ausente"
display(pd.DataFrame({"pedido": df["pedido"], "estado": estado}))
assert estado.tolist() == ["válido", "válido", "formato incorrecto", "ausente", "formato incorrecto"]

,fecha,nivel,mensaje
0,2026-10-12,INFO,Inicio
1,2026-10-12,ERROR,Fallo BD
2,NaT,WARN,Revisar fecha
3,NaT,<NA>,<NA>
4,NaT,<NA>,<NA>


nivel
INFO     1
ERROR    1
WARN     1
Name: count, dtype: Int64

,pedido,estado
0,PED-2026-0001,válido
1,PED-2026-0002,válido
2,ped-2026-0003,formato incorrecto
3,<NA>,ausente
4,PED-26-0005,formato incorrecto


## 20. Referencia rápida y documentación

| Necesito… | Método |
|---|---|
| Filtrar por una coincidencia | `str.contains(patron, na=False)` |
| Comprobar un comienzo | `str.match(patron, na=False)` |
| Validar todo el campo | `str.fullmatch(patron, na=False)` |
| Extraer la primera coincidencia | `str.extract(patron_con_grupos)` |
| Extraer todas en filas | `str.extractall(patron_con_grupos)` |
| Extraer todas en listas | `str.findall(patron)` |
| Contar coincidencias | `str.count(patron)` |
| Sustituir coincidencias | `str.replace(patron, reemplazo, regex=True)` |
| Separar por patrón | `str.split(patron, regex=True)` |

Documentación oficial consultada el 12/10/2026:

- [Pandas: trabajo con texto](https://pandas.pydata.org/docs/user_guide/text.html)
- [Python: sintaxis y comportamiento de re](https://docs.python.org/3/library/re.html)
- [Pandas: contains](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html)
- [Pandas: extract](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.extract.html)
- [Pandas: extractall](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.extractall.html)
- [Pandas: replace](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.replace.html)

Los datos y ejercicios han sido creados para este manual. La versión usada al ejecutar el cuaderno aparece en la primera celda.